In [1]:
import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

In [2]:
df=pd.read_csv("dataset/jan to may police violation_anonymized791b166.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (298450, 24)
Columns: ['id', 'latitude', 'longitude', 'location', 'vehicle_number', 'vehicle_type', 'description', 'violation_type', 'offence_code', 'created_datetime', 'closed_datetime', 'modified_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name', 'action_taken_timestamp', 'data_sent_to_scita_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'validation_status', 'validation_timestamp']


In [3]:
# These were 100% missing in EDA — dead weight, dropping immediately
drop_empty = ['description', 'closed_datetime', 'action_taken_timestamp']
df = df.drop(columns=drop_empty)
print("Shape after dropping empty columns:", df.shape)

Shape after dropping empty columns: (298450, 21)


In [4]:
# Remove confirmed invalid violations — these are noise, not signal
# Missing validation_status is OK to keep (system rollout gap, not invalid)
df_clean = df[~df['validation_status'].isin(['rejected', 'duplicate'])].copy()
print("Shape after filtering invalid records:", df_clean.shape)

Shape after filtering invalid records: (248376, 21)


In [5]:
print("Remaining nulls:")
print(df_clean.isnull().sum())

Remaining nulls:
id                                   0
latitude                             0
longitude                            0
location                          2770
vehicle_number                       0
vehicle_type                         0
violation_type                       0
offence_code                         0
created_datetime                     0
modified_datetime                    0
device_id                            0
created_by_id                        5
center_code                       9538
police_station                       5
data_sent_to_scita                   0
junction_name                        5
data_sent_to_scita_timestamp    208926
updated_vehicle_number          125254
updated_vehicle_type            125254
validation_status               125254
validation_timestamp            125254
dtype: int64


In [6]:
#data_sent_to_scita_timestamp: 84% missing, just a logging timestamp, not predictive
#updated_vehicle_number, updated_vehicle_type: only useful for vehicle-level lookup, 
#not for junction/time-based congestion prediction
#validation_timestamp: logging artifact, not predictive
drop_cols_2 = ['data_sent_to_scita_timestamp', 'updated_vehicle_number', 
               'updated_vehicle_type', 'validation_timestamp']
df_clean = df_clean.drop(columns=drop_cols_2)
print("Shape after dropping:", df_clean.shape)

Shape after dropping: (248376, 17)


In [7]:
df_clean['validation_status'] = df_clean['validation_status'].fillna('unvalidated')
df_clean = df_clean.drop(columns=['location'])

In [8]:
print(df_clean.groupby('police_station')['center_code'].nunique().describe())

count    54.000000
mean      0.962963
std       0.190626
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: center_code, dtype: float64


In [9]:
df_clean = df_clean.dropna(subset=['created_by_id', 'police_station', 'junction_name'])
print("Shape after dropping 5 problematic rows:", df_clean.shape)

Shape after dropping 5 problematic rows: (248371, 16)


In [10]:
df_clean = df_clean.drop(columns=['center_code'])
print("Shape after dropping center_code:", df_clean.shape)
print("\nRemaining nulls:")
print(df_clean.isnull().sum())

Shape after dropping center_code: (248371, 15)

Remaining nulls:
id                    0
latitude              0
longitude             0
vehicle_number        0
vehicle_type          0
violation_type        0
offence_code          0
created_datetime      0
modified_datetime     0
device_id             0
created_by_id         0
police_station        0
data_sent_to_scita    0
junction_name         0
validation_status     0
dtype: int64


In [11]:
def parse_list(x):
    try:
        return ast.literal_eval(x)
    except:
        return [x]
df_clean['violation_list'] = df_clean['violation_type'].apply(parse_list)
df_clean['primary_violation'] = df_clean['violation_list'].apply(
    lambda x: x[0] if len(x) > 0 else 'UNKNOWN'
)
df_clean['violation_count_per_row'] = df_clean['violation_list'].apply(len)
print("Primary violation done")
print("Unique primary violations:", df_clean['primary_violation'].nunique())

Primary violation done
Unique primary violations: 17


In [12]:
df_clean['created_datetime'] = pd.to_datetime(df_clean['created_datetime'])

df_clean['hour']           = df_clean['created_datetime'].dt.hour
df_clean['day_of_week']    = df_clean['created_datetime'].dt.day_name()
df_clean['day_of_week_num']= df_clean['created_datetime'].dt.dayofweek
df_clean['month']          = df_clean['created_datetime'].dt.month
df_clean['date']           = df_clean['created_datetime'].dt.date
df_clean['is_weekend']     = df_clean['day_of_week_num'].isin([5,6]).astype(int)

# Cyclical hour encoding — important since violations are concentrated 
# late night/early morning (hour 23 and hour 0 are neighbors)
df_clean['hour_sin'] = np.sin(2 * np.pi * df_clean['hour'] / 24)
df_clean['hour_cos'] = np.cos(2 * np.pi * df_clean['hour'] / 24)

print("Temporal features done")
print(df_clean[['hour','day_of_week','month','is_weekend']].nunique())

Temporal features done
hour           24
day_of_week     7
month           6
is_weekend      2
dtype: int64


In [13]:
agg = df_clean.groupby(['junction_name', 'date', 'hour']).size().reset_index(name='violation_count')
print("Aggregated shape:", agg.shape)
print(agg['violation_count'].describe())

Aggregated shape: (31429, 4)
count    31429.000000
mean         7.902606
std         17.194876
min          1.000000
25%          1.000000
50%          2.000000
75%          6.000000
max        268.000000
Name: violation_count, dtype: float64


In [14]:
agg['date'] = pd.to_datetime(agg['date'])
agg['day_of_week_num'] = agg['date'].dt.dayofweek
agg['month'] = agg['date'].dt.month
agg['is_weekend'] = agg['day_of_week_num'].isin([5,6]).astype(int)

agg['hour_sin'] = np.sin(2 * np.pi * agg['hour'] / 24)
agg['hour_cos'] = np.cos(2 * np.pi * agg['hour'] / 24)

print("Temporal features added to agg")
print(agg.shape)

Temporal features added to agg
(31429, 9)


In [15]:
junction_stats = df_clean.groupby('junction_name').agg(
    junction_total_violations=('id', 'count'),
    junction_unique_days=('date', 'nunique')
).reset_index()
junction_stats['junction_avg_per_day'] = (
    junction_stats['junction_total_violations'] / junction_stats['junction_unique_days']
)

agg = agg.merge(junction_stats, on='junction_name', how='left')
print("Junction stats merged")
print(agg.shape)

Junction stats merged
(31429, 12)


In [16]:
junction_coords = df_clean.groupby('junction_name')[['latitude','longitude']].mean().reset_index()
agg = agg.merge(junction_coords, on='junction_name', how='left')

print("Final shape:", agg.shape)
print("Columns:", agg.columns.tolist())
print("Nulls:", agg.isnull().sum().sum())

Final shape: (31429, 14)
Columns: ['junction_name', 'date', 'hour', 'violation_count', 'day_of_week_num', 'month', 'is_weekend', 'hour_sin', 'hour_cos', 'junction_total_violations', 'junction_unique_days', 'junction_avg_per_day', 'latitude', 'longitude']
Nulls: 0


In [17]:
from sklearn.model_selection import KFold
features_no_te = [
    'junction_name',
    'hour',
    'day_of_week_num',
    'month',
    'is_weekend',
    'hour_sin',
    'hour_cos',
    'junction_total_violations',
    'junction_unique_days',
    'junction_avg_per_day',
    'latitude',
    'longitude'
]
X_no_te = agg[features_no_te].copy()
y_no_te = np.log1p(agg['violation_count'])
print("X shape:", X_no_te.shape)
print("y shape:", y_no_te.shape)
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
scores = []
for train_idx, val_idx in kf.split(X_no_te):
    X_train = X_no_te.iloc[train_idx]
    X_val = X_no_te.iloc[val_idx]
    y_train = y_no_te.iloc[train_idx]
    y_val = y_no_te.iloc[val_idx]
    model = CatBoostRegressor(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=0
    )
    model.fit(
        X_train,
        y_train,
        cat_features=['junction_name']
    )
    preds = model.predict(X_val)
    scores.append(
        r2_score(y_val, preds)
    )
print(
    f"CatBoost NO-TE CV R2: {np.mean(scores):.4f} ± {np.std(scores):.4f}"
)

X shape: (31429, 12)
y shape: (31429,)
CatBoost NO-TE CV R2: 0.5551 ± 0.0109


In [18]:
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import cross_val_score

dummy = DummyRegressor(strategy="mean")

dummy_scores = cross_val_score(
    dummy,
    X_no_te,
    y_no_te,
    cv=5,
    scoring="r2"
)

print(
    f"Dummy R2: {dummy_scores.mean():.4f} ± {dummy_scores.std():.4f}"
)

Dummy R2: -0.1923 ± 0.2301


In [19]:
features_temporal = [
    'junction_name',
    'hour',
    'day_of_week_num',
    'month',
    'is_weekend',
    'hour_sin',
    'hour_cos',
    'junction_total_violations',
    'junction_unique_days',
    'junction_avg_per_day',
    'latitude',
    'longitude'
]
# Sort by date
agg_sorted = agg.sort_values('date').reset_index(drop=True)
X_temp = agg_sorted[features_temporal]
y_temp = np.log1p(agg_sorted['violation_count'])
# First 80% dates = train
# Last 20% dates = validation
split_idx = int(len(agg_sorted) * 0.8)
X_train = X_temp.iloc[:split_idx]
X_val = X_temp.iloc[split_idx:]
y_train = y_temp.iloc[:split_idx]
y_val = y_temp.iloc[split_idx:]
print("Train Rows:", len(X_train))
print("Validation Rows:", len(X_val))
model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=0
)
model.fit(
    X_train,
    y_train,
    cat_features=['junction_name']
)
preds = model.predict(X_val)
r2 = r2_score(y_val, preds)
print(f"\nTemporal Holdout R2: {r2:.4f}")

Train Rows: 25143
Validation Rows: 6286

Temporal Holdout R2: 0.5436


In [20]:
features = [
    'junction_name',
    'hour',
    'day_of_week_num',
    'month',
    'is_weekend',
    'hour_sin',
    'hour_cos',
    'junction_total_violations',
    'junction_unique_days',
    'junction_avg_per_day',
    'latitude',
    'longitude'
]
X = agg[features]
y = np.log1p(agg['violation_count'])
final_model = CatBoostRegressor(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    loss_function='RMSE',
    verbose=100
)
final_model.fit(
    X,
    y,
    cat_features=['junction_name']
)
print("Final Model Training Complete")

0:	learn: 0.9382092	total: 73.8ms	remaining: 36.8s
100:	learn: 0.6470219	total: 7.45s	remaining: 29.4s
200:	learn: 0.6376739	total: 14.7s	remaining: 21.9s
300:	learn: 0.6321480	total: 22.2s	remaining: 14.7s
400:	learn: 0.6282779	total: 29.6s	remaining: 7.3s
499:	learn: 0.6249544	total: 37.1s	remaining: 0us
Final Model Training Complete


In [21]:
agg['predicted_log'] = final_model.predict(X)

agg['predicted_violations'] = np.expm1(
    agg['predicted_log']
)

print(
    agg['predicted_violations'].describe()
)

count    31429.000000
mean         6.186015
std         12.749740
min          0.909159
25%          1.997348
50%          2.686890
75%          3.986884
max        104.498763
Name: predicted_violations, dtype: float64


In [22]:
preds = final_model.predict(X.head(10))
print(preds)

[1.46930954 1.3500127  1.17643871 1.39895144 1.62853566 1.49712786
 1.51293438 1.53491948 1.48634469 1.21473666]


In [23]:
preds_1000 = final_model.predict(X.head(1000))
print(len(preds_1000))

1000


In [24]:
preds_10000 = final_model.predict(X.head(10000))
print(len(preds_10000))

10000


In [25]:
preds_all = final_model.predict(X)
print(len(preds_all))

31429


In [26]:
print(X.dtypes)

junction_name                 object
hour                           int32
day_of_week_num                int32
month                          int32
is_weekend                     int64
hour_sin                     float64
hour_cos                     float64
junction_total_violations      int64
junction_unique_days           int64
junction_avg_per_day         float64
latitude                     float64
longitude                    float64
dtype: object


In [27]:
agg['predicted_log'] = preds_all
print("Assigned")

Assigned


In [28]:
agg['predicted_violations'] = np.expm1(
    agg['predicted_log']
)
print("Converted")

Converted


In [29]:
print(
    agg['predicted_violations'].describe()
)

count    31429.000000
mean         6.186015
std         12.749740
min          0.909159
25%          1.997348
50%          2.686890
75%          3.986884
max        104.498763
Name: predicted_violations, dtype: float64


In [30]:
agg['risk_level'] = pd.qcut(
    agg['predicted_violations'],
    q=[0, 0.25, 0.50, 0.80, 1.0],
    labels=['Low', 'Medium', 'High', 'Critical']
)

print(
    agg['risk_level'].value_counts()
)

risk_level
High        9428
Low         7858
Medium      7857
Critical    6286
Name: count, dtype: int64


In [31]:
print(
    agg.groupby('risk_level')['predicted_violations']
       .describe()
)

             count       mean        std       min       25%       50%  \
risk_level                                                               
Low         7858.0   1.696699   0.193533  0.909159  1.552960  1.712920   
Medium      7857.0   2.301274   0.194186  1.997349  2.127609  2.288031   
High        9428.0   3.409068   0.509599  2.687310  2.949022  3.346448   
Critical    6286.0  20.818613  23.293664  4.521477  5.346011  6.833316   

                  75%         max  
risk_level                         
Low          1.862418    1.997348  
Medium       2.454990    2.686890  
High         3.812661    4.520968  
Critical    37.491652  104.498763  


In [32]:
top_hotspots = (
    agg.sort_values(
        'predicted_violations',
        ascending=False
    )
    .head(20)
)

print(
    top_hotspots[
        [
            'junction_name',
            'hour',
            'predicted_violations',
            'risk_level'
        ]
    ]
)

      junction_name  hour  predicted_violations risk_level
30661   No Junction     5            104.498763   Critical
30539   No Junction     5            104.498763   Critical
30417   No Junction     5            104.498763   Critical
30289   No Junction     5            104.498763   Critical
31405   No Junction     5            103.455782   Critical
30909   No Junction     5            101.659552   Critical
31026   No Junction     5            101.659552   Critical
31283   No Junction     5            101.659552   Critical
30784   No Junction     5            101.659552   Critical
31151   No Junction     5            101.659552   Critical
30175   No Junction     5            100.702437   Critical
29826   No Junction     5            100.702437   Critical
30058   No Junction     5            100.702437   Critical
29940   No Junction     5            100.702437   Critical
30540   No Junction     6             92.576062   Critical
30662   No Junction     6             92.576062   Critic

Output skewed so interaction between junction_name,hour.predicted_violations,risk_level is failed.

In [33]:
print(
    agg['predicted_violations']
      .quantile([0.80, 0.90, 0.95, 0.99])
)

0.80     4.521171
0.90     6.833289
0.95    37.491652
0.99    70.351548
Name: predicted_violations, dtype: float64


In [34]:
dashboard_df = agg[
    agg['junction_name'] != 'No Junction'
].copy()

top_hotspots = (
    dashboard_df
    .sort_values(
        'predicted_violations',
        ascending=False
    )
    .head(20)
)

print(
    top_hotspots[
        [
            'junction_name',
            'hour',
            'predicted_violations'
        ]
    ]
)

                        junction_name  hour  predicted_violations
10506  BTP051 - Safina Plaza Junction     4             14.006246
10440  BTP051 - Safina Plaza Junction     4             14.006246
10319  BTP051 - Safina Plaza Junction     4             14.006246
10379  BTP051 - Safina Plaza Junction     4             14.006246
10380  BTP051 - Safina Plaza Junction     5             13.741552
10320  BTP051 - Safina Plaza Junction     5             13.741552
10441  BTP051 - Safina Plaza Junction     5             13.741552
10507  BTP051 - Safina Plaza Junction     5             13.741552
11108  BTP051 - Safina Plaza Junction     4             13.695022
10726  BTP051 - Safina Plaza Junction     4             13.686636
10616  BTP051 - Safina Plaza Junction     4             13.686636
10672  BTP051 - Safina Plaza Junction     4             13.686636
10559  BTP051 - Safina Plaza Junction     4             13.686636
11040  BTP051 - Safina Plaza Junction     4             13.629519
10963  BTP

Output skewed so interaction between junction_name,hour.predicted_violations is failed.

In [35]:
print('location' in df.columns)

True


In [36]:
no_junction_raw = df[
    df['junction_name'] == 'No Junction'
]

print("Rows:", len(no_junction_raw))
print("Missing location:", no_junction_raw['location'].isna().sum())

print(
    "\nPercent with location:",
    round(
        100 * no_junction_raw['location'].notna().mean(),
        2
    ),
    "%"
)

Rows: 147880
Missing location: 1704

Percent with location: 98.85 %


In [37]:
print(
    no_junction_raw['location']
    .value_counts()
    .head(20)
)

location
Unnamed Road, Begur Chikkanahalli, Bengaluru, Karnataka. Pin-562149 (India)                                                4090
New Horizon College Road, New Horizon College of Engineering, Kadubisanahalli, Bengaluru, Karnataka. Pin-560103 (India)    3785
MBT Road, Devasandra Junction, KR Puram, Bengaluru, Karnataka. Pin-560036 (India)                                          3027
Bellary Road, Vinayaka Nagar, Hebbal, Bengaluru, Karnataka. Pin-560024 (India)                                             2639
New Horizon College Road, Embassy Tech Village, Devara Beesana Halli, Bengaluru, Karnataka. Pin-560103 (India)             2416
80 Feet Ring Road, Orion, Brigade Gateway, Malleshwaram West, Bengaluru, Karnataka. Pin-560055 (India)                     2115
Sahakar Nagar Road, Fortuna Acacia, Byatarayanapura, Bengaluru, Karnataka. Pin-560092 (India)                              1743
Sahakar Nagar Road, Fortune Acacia, Byatarayanapura, Bengaluru, Karnataka. Pin-560092 (India)  

Well seprated output hence location_summary can be taken as a main separating parameter.

In [38]:
location_map = (
    df.groupby('junction_name')['location']
      .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
      .reset_index()
)

print(location_map.shape)

(169, 2)


In [39]:
dashboard_df = agg.merge(
    location_map,
    on='junction_name',
    how='left'
)

print(dashboard_df.shape)

(31429, 18)


In [40]:
dashboard_df['display_location'] = np.where(
    dashboard_df['junction_name'] == 'No Junction',
    dashboard_df['location'],
    dashboard_df['junction_name']
)

print(
    dashboard_df['display_location']
    .isna()
    .sum()
)

0


In [41]:
dashboard_df['risk_level'] = pd.cut(
    dashboard_df['predicted_violations'],
    bins=[0, 2, 4.5, 6.8, 37.5, np.inf],
    labels=[
        'Low',
        'Medium',
        'High',
        'Critical',
        'Extreme'
    ]
)

print(
    dashboard_df['risk_level']
    .value_counts()
)

risk_level
Medium      17204
Low          7908
High         3144
Critical     1603
Extreme      1570
Name: count, dtype: int64


In [42]:
top_hotspots = (
    dashboard_df
    .sort_values(
        'predicted_violations',
        ascending=False
    )
)

print(
    top_hotspots[
        [
            'display_location',
            'hour',
            'predicted_violations',
            'risk_level'
        ]
    ].head()
)

                                        display_location  hour  \
30661  Unnamed Road, Begur Chikkanahalli, Bengaluru, ...     5   
30539  Unnamed Road, Begur Chikkanahalli, Bengaluru, ...     5   
30417  Unnamed Road, Begur Chikkanahalli, Bengaluru, ...     5   
30289  Unnamed Road, Begur Chikkanahalli, Bengaluru, ...     5   
31405  Unnamed Road, Begur Chikkanahalli, Bengaluru, ...     5   

       predicted_violations risk_level  
30661            104.498763    Extreme  
30539            104.498763    Extreme  
30417            104.498763    Extreme  
30289            104.498763    Extreme  
31405            103.455782    Extreme  


In [43]:
print(
    dashboard_df['display_location']
    .nunique()
)
print(
    dashboard_df['display_location']
    .isna()
    .sum()
)

169
0


In [44]:
location_summary = (
    dashboard_df.groupby('display_location')
    .agg(
        avg_predicted_violations=('predicted_violations', 'mean'),
        max_predicted_violations=('predicted_violations', 'max'),
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first')
    )
    .reset_index()
)

print(location_summary.shape)

(169, 5)


In [45]:
location_summary['risk_level'] = pd.cut(
    location_summary['avg_predicted_violations'],
    bins=[0, 2, 4.5, 6.8, 37.5, np.inf],
    labels=[
        'Low',
        'Medium',
        'High',
        'Critical',
        'Extreme'
    ]
)

In [46]:
print(
    location_summary['risk_level']
    .value_counts()
)

risk_level
Low         94
Medium      65
High         9
Extreme      1
Critical     0
Name: count, dtype: int64


In [47]:
print(
    location_summary['avg_predicted_violations']
    .describe()
)
print(
    location_summary['avg_predicted_violations']
    .quantile([0.50, 0.80, 0.90, 0.95, 0.99])
)

count    169.000000
mean       2.450046
std        3.168643
min        1.180302
25%        1.618444
50%        1.917578
75%        2.479494
max       41.569849
Name: avg_predicted_violations, dtype: float64
0.50    1.917578
0.80    2.630795
0.90    3.698361
0.95    4.566657
0.99    5.633994
Name: avg_predicted_violations, dtype: float64


In [48]:
location_summary['risk_level'] = pd.qcut(
    location_summary['avg_predicted_violations'],
    q=[0, 0.5, 0.8, 0.95, 0.99, 1.0],
    labels=[
        'Low',
        'Medium',
        'High',
        'Critical',
        'Extreme'
    ]
)

print(location_summary['risk_level'].value_counts())

risk_level
Low         85
Medium      50
High        25
Critical     7
Extreme      2
Name: count, dtype: int64


In [49]:
print(location_summary.columns.tolist())
print(location_summary.shape)

['display_location', 'avg_predicted_violations', 'max_predicted_violations', 'latitude', 'longitude', 'risk_level']
(169, 6)


In [50]:
# ── Blocking Severity Weights ────────────────────────────────
vehicle_weight = {
    'LORRY/GOODS VEHICLE': 3.0, 'BUS (BMTC/KSRTC)': 3.0, 'TOURIST BUS': 3.0,
    'PRIVATE BUS': 3.0, 'FACTORY BUS': 3.0, 'TANKER': 3.0, 'HGV': 3.0,
    'MINI LORRY': 2.5, 'TEMPO': 2.0, 'LGV': 2.0, 'VAN': 1.8,
    'CAR': 1.5, 'JEEP': 1.5, 'MAXI-CAB': 1.5, 'GOODS AUTO': 1.5,
    'PASSENGER AUTO': 1.2, 'SCHOOL VEHICLE': 1.5, 'TRACTOR': 2.0,
    'SCOOTER': 1.0, 'MOTOR CYCLE': 1.0, 'MOPED': 1.0, 'OTHERS': 1.2
}

violation_weight = {
    'DOUBLE PARKING': 2.5, 'PARKING IN A MAIN ROAD': 2.0,
    'PARKING NEAR ROAD CROSSING': 2.0,
    'PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS': 2.0,
    'AGAINST ONE WAY/NO ENTRY': 1.8, 'WRONG PARKING': 1.5,
    'NO PARKING': 1.3, 'PARKING ON FOOTPATH': 1.2,
    'PARKING OPPOSITE TO ANOTHER PARKED VEHICLE': 1.5,
    'PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC': 1.5
}

df_clean['vehicle_weight']    = df_clean['vehicle_type'].map(vehicle_weight).fillna(1.0)
df_clean['violation_weight']  = df_clean['primary_violation'].map(violation_weight).fillna(1.0)
df_clean['blocking_severity'] = df_clean['vehicle_weight'] * df_clean['violation_weight']

# Group by junction_name since display_location doesn't exist in df_clean
severity_by_junction = df_clean.groupby('junction_name')['blocking_severity'].mean()

# Map to location_summary via display_location 
# (display_location = junction_name for named junctions in your current setup)
location_summary['avg_blocking_severity'] = (
    location_summary['display_location'].map(severity_by_junction)
)

# Fill any unmapped (No Junction rows) with overall mean
overall_severity = df_clean['blocking_severity'].mean()
location_summary['avg_blocking_severity'] = (
    location_summary['avg_blocking_severity'].fillna(overall_severity)
)

location_summary['congestion_impact_score'] = (
    location_summary['avg_blocking_severity'].rank(pct=True) * 100
).round(1)

# ── Officers and Tow Trucks ──────────────────────────────────
officer_map   = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 5, 'Extreme': 8}
tow_truck_map = {'Low': 0, 'Medium': 1, 'High': 1, 'Critical': 2, 'Extreme': 3}

location_summary['officers']   = location_summary['risk_level'].map(officer_map)
location_summary['tow_trucks'] = location_summary['risk_level'].map(tow_truck_map)


In [51]:
print(location_summary['risk_level'].value_counts().reindex(
    ['Low','Medium','High','Critical','Extreme']))
print(location_summary['avg_predicted_violations'].describe())

risk_level
Low         85
Medium      50
High        25
Critical     7
Extreme      2
Name: count, dtype: int64
count    169.000000
mean       2.450046
std        3.168643
min        1.180302
25%        1.618444
50%        1.917578
75%        2.479494
max       41.569849
Name: avg_predicted_violations, dtype: float64


In [52]:
import os
os.makedirs('../data', exist_ok=True)

location_summary.to_csv('../data/location_summary.csv', index=False)
dashboard_df.to_csv('../data/dashboard_predictions.csv', index=False)

print("Exported successfully")
print("location_summary shape:", location_summary.shape)
print("dashboard_df shape:", dashboard_df.shape)
print("Saved to:", os.path.abspath('../data'))

Exported successfully
location_summary shape: (169, 10)
dashboard_df shape: (31429, 19)
Saved to: C:\Users\rathp\OneDrive\Desktop\Flipkart Gridlock 2.0\Flipkart Gridlock 2.0 Theme poor Visibility on Parking-Induced Congestion\data


In [53]:
import os

os.makedirs('../model', exist_ok=True)

# Save using CatBoost native format only — no need for joblib
final_model.save_model('../model/parking_congestion_model.cbm')

print("Model saved to:", os.path.abspath('../model'))

Model saved to: C:\Users\rathp\OneDrive\Desktop\Flipkart Gridlock 2.0\Flipkart Gridlock 2.0 Theme poor Visibility on Parking-Induced Congestion\model
